# CS5760 NLP — Homework 1
Name:RISHIK VARDHAN REDDY UMMENTHALA  
Student ID:700777149

In [2]:
# Q1 — Regular Expressions
import re

# 1.1 — U.S. ZIP codes
# \b = word boundary (prevents matching inside longer numbers like 123456)
# \d{5} = exactly 5 digits
# (?:[-\s]\d{4})? = optionally followed by hyphen or space + 4 digits
zip_pattern = r'\b\d{5}(?:[-\s]\d{4})?\b'

test_zip = "Valid: 12345  12345-6789  12345 6789   Invalid: 123456  1234"
print("Q1.1 — ZIP codes:")
print(re.findall(zip_pattern, test_zip))

Q1.1 — ZIP codes:
['12345', '12345-6789', '12345 6789']


In [3]:
# 1.2 — Words NOT starting with a capital letter
# (?![A-Z]) = negative lookahead — skip words starting with uppercase
# [a-z] = first character must be lowercase
# [a-zA-Z'\-]* = rest can include letters, apostrophes, hyphens
non_cap_pattern = r"\b(?![A-Z])[a-z][a-zA-Z'\-]*\b"

test_words = "The quick brown fox. don't state-of-the-art Hello World"
print("Q1.2 — Words not starting with a capital:")
print(re.findall(non_cap_pattern, test_words))

Q1.2 — Words not starting with a capital:
['quick', 'brown', 'fox', "don't", 'state-of-the-art']


In [4]:
# 1.3 — Rich numbers (sign, thousands separators, decimals, scientific notation)
# [+-]?          = optional sign
# \d{1,3}        = 1-3 leading digits
# (?:,\d{3})*    = zero or more comma-separated thousands groups
# (?:\.\d+)?     = optional decimal part
# (?:[eE][+-]?\d+)? = optional scientific notation
num_pattern = r'[+-]?\d{1,3}(?:,\d{3})*(?:\.\d+)?(?:[eE][+-]?\d+)?'

test_nums = "Numbers: +1,234.56  -0.5  1.23e-4  42  1,000,000  3.14"
print("Q1.3 — Rich numbers:")
print(re.findall(num_pattern, test_nums))

Q1.3 — Rich numbers:
['+1,234.56', '-0.5', '1.23e-4', '42', '1,000,000', '3.14']


In [5]:
# 1.4 — Email spelling variants (case-insensitive)
# \b = word boundary
# e = literal e
# [\s\-\u2013]? = optional space, hyphen, or en-dash (–)
# mail = literal mail
# re.IGNORECASE handles E-Mail, EMAIL etc.
email_pattern = r'\be[\s\-\u2013]?mail\b'

test_email = "email  e-mail  e mail  E-Mail  E MAIL  e–mail"
print("Q1.4 — Email variants:")
print(re.findall(email_pattern, test_email, re.IGNORECASE))

Q1.4 — Email variants:
['email', 'e-mail', 'e mail', 'E-Mail', 'E MAIL', 'e–mail']


In [7]:
# 1.5 — Interjection go / goo / gooo…
# \b = word boundary (won't match inside "goes")
# go+ = literal g followed by ONE or more o's
# [!.,?]? = optional single trailing punctuation
go_pattern = r'\bgo+[!.,?]?(?=\s|$)'

test_go = "go  goo!  gooo.  goooo,  gooooo?  Stop  goes"
print("Q1.5 — go/goo/gooo:")
print(re.findall(go_pattern, test_go))

Q1.5 — go/goo/gooo:
['go', 'goo!', 'gooo.', 'goooo,', 'gooooo?']


In [8]:
# 1.6 — Lines ending with a question mark
# \? = literal question mark
# [\)\]"'`]* = zero or more closing quotes/brackets after ?
# \s*$ = optional whitespace then end of line
line_end_pattern = r'\?[\)\]"\'`]*\s*$'

test_lines = """Is this a question?
What happened?"]
No.
Really?']
Just a statement"""

print("Q1.6 — Lines ending with ?:")
for line in test_lines.splitlines():
    if re.search(line_end_pattern, line):
        print(f"  MATCH: {repr(line)}")

Q1.6 — Lines ending with ?:
  MATCH: 'Is this a question?'
  MATCH: 'What happened?"]'
  MATCH: "Really?']"


## Q2.1 — Manual BPE on a Toy Corpus

**Corpus:** low×5, lowest×2, newer×6, wider×3, new×2

**Initial vocabulary (character-level + end marker `_`):**

V₀ = {_, d, e, i, l, n, o, r, s, t, w}

| Token | Frequency |
|---|---|
| l o w _ | 5 |
| l o w e s t _ | 2 |
| n e w e r _ | 6 |
| w i d e r _ | 3 |
| n e w _ | 2 |

**Step 1:** Most frequent pair: **(e, r) → er** (count=9)

**Step 2:** Most frequent pair: **(er, _) → er_** (count=9)

**Step 3:** Most frequent pair: **(n, e) → ne** (count=8)

In [9]:
# Q2.2 — Mini BPE Learner
from collections import defaultdict, Counter

CORPUS = (
    "low low low low low "
    "lowest lowest "
    "newer newer newer newer newer newer "
    "wider wider wider "
    "new new"
)

def build_initial_vocab(corpus):
    """Split each word into characters and add end-of-word marker _"""
    word_counts = Counter(corpus.split())
    vocab = {}
    for word, count in word_counts.items():
        token = ' '.join(list(word)) + ' _'
        vocab[token] = count
    return vocab

def get_bigram_counts(vocab):
    """Count every adjacent symbol pair weighted by word frequency"""
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

def merge_vocab(pair, vocab):
    """Replace every occurrence of pair with the merged token"""
    bigram = ' '.join(pair)
    merged = ''.join(pair)
    return {word.replace(bigram, merged): freq for word, freq in vocab.items()}

def learn_bpe(corpus, num_merges):
    vocab = build_initial_vocab(corpus)
    merges = []

    print("Initial vocab:")
    for word, count in sorted(vocab.items()):
        print(f"  '{word}' x{count}")

    for step in range(1, num_merges + 1):
        pairs = get_bigram_counts(vocab)
        if not pairs:
            break
        best_pair = max(pairs, key=pairs.get)
        vocab = merge_vocab(best_pair, vocab)
        merges.append(best_pair)
        print(f"\nStep {step}: merge {best_pair} → '{''.join(best_pair)}'")

    return vocab, merges

vocab, merges = learn_bpe(CORPUS, num_merges=3)

Initial vocab:
  'l o w _' x5
  'l o w e s t _' x2
  'n e w _' x2
  'n e w e r _' x6
  'w i d e r _' x3

Step 1: merge ('e', 'r') → 'er'

Step 2: merge ('er', '_') → 'er_'

Step 3: merge ('n', 'e') → 'ne'


In [10]:
# Segmenting words using learned merges
def apply_bpe(word, merges):
    """Replay learned merges on a new word"""
    symbols = list(word) + ['_']
    for pair in merges:
        i = 0
        new_symbols = []
        while i < len(symbols):
            if (i < len(symbols) - 1
                    and symbols[i] == pair[0]
                    and symbols[i+1] == pair[1]):
                new_symbols.append(''.join(pair))
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols
    return symbols

print("Word segmentations:")
test_words = ["new", "newer", "lowest", "widest", "newestest"]
for word in test_words:
    seg = apply_bpe(word, merges)
    print(f"  {word:>12} → {seg}")

Word segmentations:
           new → ['ne', 'w', '_']
         newer → ['ne', 'w', 'er_']
        lowest → ['l', 'o', 'w', 'e', 's', 't', '_']
        widest → ['w', 'i', 'd', 'e', 's', 't', '_']
     newestest → ['ne', 'w', 'e', 's', 't', 'e', 's', 't', '_']


## Q2.2 — Observations

- **OOV handling:** BPE breaks unseen words into smaller known pieces instead 
of treating them as completely unknown. For example, `widest` was never in 
the corpus but its characters were — so it still gets a representation.

- **Morphemic alignment:** The merge `er_` aligns with the English comparative 
suffix *-er* (newer, wider), meaning the model naturally learns a meaningful 
linguistic unit just from frequency.

In [11]:
# Q2.3 — BPE on a Telugu paragraph
TELUGU_CORPUS = (
    "నేను పాఠశాలకు వెళ్తాను నేను పుస్తకాలు చదువుతాను "
    "పాఠశాల చాలా పెద్దది పుస్తకాలు చాలా బాగున్నాయి "
    "నేను రోజూ పాఠశాలకు వెళ్తాను చదువు చాలా ముఖ్యమైనది"
)

print("Telugu paragraph used:")
print(TELUGU_CORPUS)
print()

vocab_telugu, merges_telugu = learn_bpe(TELUGU_CORPUS, num_merges=10)

print("\nFive most frequent merges (in order learned):")
for i, m in enumerate(merges_telugu[:5], 1):
    print(f"  {i}. {m[0]} + {m[1]} → {''.join(m)}")

print("\nSegmentation of 5 words:")
seg_words = ["పాఠశాల", "పుస్తకాలు", "చదువుతాను", "వెళ్తాను", "ముఖ్యమైనది"]
for word in seg_words:
    seg = apply_bpe(word, merges_telugu)
    print(f"  {word} → {seg}")

Telugu paragraph used:
నేను పాఠశాలకు వెళ్తాను నేను పుస్తకాలు చదువుతాను పాఠశాల చాలా పెద్దది పుస్తకాలు చాలా బాగున్నాయి నేను రోజూ పాఠశాలకు వెళ్తాను చదువు చాలా ముఖ్యమైనది

Initial vocab:
  'చ ద ు వ ు _' x1
  'చ ద ు వ ు త ా న ు _' x1
  'చ ా ల ా _' x3
  'న ే న ు _' x3
  'ప ా ఠ శ ా ల _' x1
  'ప ా ఠ శ ా ల క ు _' x2
  'ప ు స ్ త క ా ల ు _' x2
  'ప ె ద ్ ద ద ి _' x1
  'బ ా గ ు న ్ న ా య ి _' x1
  'మ ు ఖ ్ య మ ై న ద ి _' x1
  'ర ో జ ూ _' x1
  'వ ె ళ ్ త ా న ు _' x2

Step 1: merge ('ు', '_') → 'ు_'

Step 2: merge ('ా', 'ల') → 'ాల'

Step 3: merge ('న', 'ు_') → 'ను_'

Step 4: merge ('్', 'త') → '్త'

Step 5: merge ('న', 'ే') → 'నే'

Step 6: merge ('నే', 'ను_') → 'నేను_'

Step 7: merge ('ప', 'ా') → 'పా'

Step 8: merge ('పా', 'ఠ') → 'పాఠ'

Step 9: merge ('పాఠ', 'శ') → 'పాఠశ'

Step 10: merge ('పాఠశ', 'ాల') → 'పాఠశాల'

Five most frequent merges (in order learned):
  1. ు + _ → ు_
  2. ా + ల → ాల
  3. న + ు_ → ను_
  4. ్ + త → ్త
  5. న + ే → నే

Segmentation of 5 words:
  పాఠశాల → ['పాఠశాల', '_']
  పుస్

## Q2.3 — Reflection on Telugu BPE

**Paragraph language:** Telugu

**Observations from the 5 most frequent merges:**
- `ు_` was the first merge — the vowel ending is extremely common in Telugu, 
appearing in almost every word
- `ాల` (āla) emerged early as it appears in words like పాఠశాల (school)
- `నేను_` (nēnu = "I") was learned as a complete token by step 6, showing BPE 
can recover whole frequent words

**Segmentation analysis:**
- `పాఠశాల` was learned as a complete token — it appeared frequently enough to 
survive as one unit
- `ముఖ్యమైనది` (important) was barely seen, so it stays highly fragmented

**Pros of BPE for Telugu:**
- Handles OOV words gracefully by breaking them into known subword pieces
- Naturally learns common suffixes like `ను_` which is a grammatical ending

**Cons of BPE for Telugu:**
- Telugu script has complex conjunct consonants (e.g. `్త`, `్క`) which BPE 
splits arbitrarily based on frequency, not linguistic rules
- A small corpus like this doesn't give BPE enough data to learn meaningful 
morpheme boundaries

## Q3 — Bayes' Rule Applied to Text Classification

**Formula:** `C_MAP = argmax P(c) × P(d|c)`

---

**1. P(c) — Prior Probability**

This is how likely a class is *before* seeing any document. It is estimated 
from training data by counting how many documents belong to each class.

For example, if 60% of training documents are negative reviews, then 
P(negative) = 0.6. It represents the model's baseline belief about which 
class a random document belongs to.

---

**2. P(d|c) — Likelihood**

This is the probability of observing the document *given* that it belongs 
to class c. Under the Naïve Bayes assumption, it is computed as:

`P(w₁|c) × P(w₂|c) × … × P(wₙ|c)`

It captures how well the words in the document fit a particular class.

---

**3. P(c|d) — Posterior Probability**

This is what we actually want — the probability that the document belongs 
to class c *after* seeing its words. It combines the prior belief with the 
evidence from the document's words.

---

**4. Why P(d) can be ignored?**

When comparing classes, P(d) is the same constant for all classes since 
the document is fixed. Because we only care about which class has the 
*highest* probability, dividing by the same constant doesn't change the 
result. So we safely drop P(d) and only compare `P(c) × P(d|c)`.

## Q4 — Add-1 (Laplace) Smoothing

**Given:**
- P(−) = 3/5,  P(+) = 2/5
- Vocabulary size: V = 20
- Total tokens in negative class: N₋ = 14

---

**1. Denominator with Add-1 Smoothing**

With Laplace smoothing, we add 1 to every word count and add V to the 
denominator to keep probabilities valid:

`Denominator = N₋ + V = 14 + 20 = 34`

---

**2. P(predictable | −)**

"predictable" appears 2 times in negative documents:

`P(predictable|−) = (2 + 1) / 34 = 3/34 ≈ 0.088`

---

**3. P(fun | −)**

"fun" appears 0 times in negative documents. Without smoothing this would 
be 0, which would make the entire class probability zero. With Add-1:

`P(fun|−) = (0 + 1) / 34 = 1/34 ≈ 0.029`

**Why smoothing matters:** It ensures unseen words still get a small 
non-zero probability, preventing any single unseen word from killing 
the entire classification.

In [12]:
# Install required library
import sys
!{sys.executable} -m pip install indic-nlp-library

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 15.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [indic-nlp-library]


In [13]:
# Q5 — Tokenization on Telugu Paragraph
import re

telugu_paragraph = """ఈ రోజు భారత ప్రభుత్వం కొత్త విద్యా విధానాన్ని ప్రకటించింది. విద్యార్థులు మరియు ఉపాధ్యాయులు ఈ నిర్ణయాన్ని స్వాగతించారు. విధానం వల్ల పాఠశాలల్లో మార్పులు జరుగుతాయని భావిస్తున్నారు. అమలు వచ్చే సంవత్సరంలో ప్రారంభమవుతుంది"""

print("Original Telugu Paragraph:")
print(telugu_paragraph)
print("\n" + "="*50 + "\n")

# Q5.1a — Naive space-based tokenization
naive_tokens = telugu_paragraph.split()
print("Naive Space-based Tokens:")
print(naive_tokens)
print(f"Token count: {len(naive_tokens)}")
print("\n" + "="*50 + "\n")

# Q5.1b — Manual correction using regex
corrected_tokens = []
for word in telugu_paragraph.split():
    word = re.sub(r'[,.!?।]', '', word)
    if word.endswith('ది') and len(word) > 2:
        corrected_tokens.append(word[:-2])
        corrected_tokens.append('ది')
    elif word.endswith('లో'):
        corrected_tokens.append(word[:-2])
        corrected_tokens.append('లో')
    else:
        corrected_tokens.append(word)

corrected_tokens = [t for t in corrected_tokens if t]
print("Manually Corrected Tokens:")
print(corrected_tokens)
print(f"Token count: {len(corrected_tokens)}")

Original Telugu Paragraph:
ఈ రోజు భారత ప్రభుత్వం కొత్త విద్యా విధానాన్ని ప్రకటించింది. విద్యార్థులు మరియు ఉపాధ్యాయులు ఈ నిర్ణయాన్ని స్వాగతించారు. విధానం వల్ల పాఠశాలల్లో మార్పులు జరుగుతాయని భావిస్తున్నారు. అమలు వచ్చే సంవత్సరంలో ప్రారంభమవుతుంది


Naive Space-based Tokens:
['ఈ', 'రోజు', 'భారత', 'ప్రభుత్వం', 'కొత్త', 'విద్యా', 'విధానాన్ని', 'ప్రకటించింది.', 'విద్యార్థులు', 'మరియు', 'ఉపాధ్యాయులు', 'ఈ', 'నిర్ణయాన్ని', 'స్వాగతించారు.', 'విధానం', 'వల్ల', 'పాఠశాలల్లో', 'మార్పులు', 'జరుగుతాయని', 'భావిస్తున్నారు.', 'అమలు', 'వచ్చే', 'సంవత్సరంలో', 'ప్రారంభమవుతుంది']
Token count: 24


Manually Corrected Tokens:
['ఈ', 'రోజు', 'భారత', 'ప్రభుత్వం', 'కొత్త', 'విద్యా', 'విధానాన్ని', 'ప్రకటించిం', 'ది', 'విద్యార్థులు', 'మరియు', 'ఉపాధ్యాయులు', 'ఈ', 'నిర్ణయాన్ని', 'స్వాగతించారు', 'విధానం', 'వల్ల', 'పాఠశాలల్', 'లో', 'మార్పులు', 'జరుగుతాయని', 'భావిస్తున్నారు', 'అమలు', 'వచ్చే', 'సంవత్సరం', 'లో', 'ప్రారంభమవుతుం', 'ది']
Token count: 28


In [14]:
# Q5.2 — Compare with Indic-NLP library
from indicnlp.tokenize.indic_tokenize import trivial_tokenize

indic_tokens = [token for token in trivial_tokenize(telugu_paragraph, lang='te')]

print("Indic-NLP Tokens:")
print(indic_tokens)
print(f"Token count: {len(indic_tokens)}")
print("\n" + "="*50 + "\n")

# Comparison summary
naive_set = set(naive_tokens)
corrected_set = set(corrected_tokens)
indic_set = set(indic_tokens)

print("Tokens in Naive but NOT in Indic-NLP:")
print(sorted(naive_set - indic_set))

print("\nTokens in Indic-NLP but NOT in Naive:")
print(sorted(indic_set - naive_set))

print("\nTokens in Manual but NOT in Indic-NLP:")
print(sorted(corrected_set - indic_set))

print(f"\nTotal Naive Tokens:     {len(naive_tokens)}")
print(f"Total Manual Tokens:    {len(corrected_tokens)}")
print(f"Total Indic-NLP Tokens: {len(indic_tokens)}")

Indic-NLP Tokens:
['ఈ', 'రోజు', 'భారత', 'ప్రభుత్వం', 'కొత్త', 'విద్యా', 'విధానాన్ని', 'ప్రకటించింది', '.', 'విద్యార్థులు', 'మరియు', 'ఉపాధ్యాయులు', 'ఈ', 'నిర్ణయాన్ని', 'స్వాగతించారు', '.', 'విధానం', 'వల్ల', 'పాఠశాలల్లో', 'మార్పులు', 'జరుగుతాయని', 'భావిస్తున్నారు', '.', 'అమలు', 'వచ్చే', 'సంవత్సరంలో', 'ప్రారంభమవుతుంది']
Token count: 27


Tokens in Naive but NOT in Indic-NLP:
['ప్రకటించింది.', 'భావిస్తున్నారు.', 'స్వాగతించారు.']

Tokens in Indic-NLP but NOT in Naive:
['.', 'ప్రకటించింది', 'భావిస్తున్నారు', 'స్వాగతించారు']

Tokens in Manual but NOT in Indic-NLP:
['ది', 'పాఠశాలల్', 'ప్రకటించిం', 'ప్రారంభమవుతుం', 'లో', 'సంవత్సరం']

Total Naive Tokens:     24
Total Manual Tokens:    28
Total Indic-NLP Tokens: 27


## Q5.3 — Multiword Expressions (MWEs) in Telugu

1. **భారత ప్రభుత్వం (Bhārata Prabhutvam) — Indian Government**
   - Individual words mean "Indian" and "Government" separately
   - Together they refer specifically to the central governing authority of India
   - Splitting them causes incorrect entity recognition in NLP tasks

2. **వచ్చే సంవత్సరం (Vacchē Samvatsaram) — Next Year**
   - "వచ్చే" means "coming" and "సంవత్సరం" means "year"
   - Together they form a single temporal unit meaning "next year"
   - Splitting causes "వచ్చే" to be misread as a standalone verb

3. **విద్యా విధానం (Vidyā Vidhānam) — Education Policy**
   - "విద్యా" means "education" and "విధానం" means "policy/system"
   - Together they refer to a specific government concept
   - Separating them loses the specific administrative meaning

---

## Q5.4 — Reflection

The hardest part of Telugu tokenization is its rich morphology. Words like
పాఠశాలల్లో (in schools) pack a root, plural marker, and locative suffix
into one surface form — space-split tokenization keeps all of this glued
together as one token, losing all grammatical information.

Compared to English, Telugu is significantly harder to tokenize. English
tokenization mainly deals with punctuation and contractions. Telugu demands
handling of complex suffixes, postpositions, and conjunct consonants like
్త and ్క which attach to roots in ways that simple rules cannot fully capture.

Punctuation is relatively easy to handle — Indic-NLP correctly separates
periods as distinct tokens. Morphology is the real challenge, requiring deep
linguistic knowledge to split words into meaningful units. MWEs add another
layer of difficulty since no tokenizer can identify them without semantic
context — భారత ప్రభుత్వం looks like two ordinary words to a basic tokenizer.